In [1]:
%matplotlib qt
import numpy as np
import matplotlib.pyplot as plt
from astropy.io import fits
import glob

from processing import *
from wavelengths import *
from fit_prefilter import *
#from fit_voigt import *

In [2]:
dark_file = '/home/ulyanov/data/solo/phi/dark/solo_CAL1_phi-fdt-dark_20240205T033810_V202402220119C_0422051001.fits.gz'

In [3]:
folder = '/home/ulyanov/data/solo/phi/prefilter/calibration/2025-09-16/'
#folder = '/home/ulyanov/data/solo/phi/prefilter/calibration/2024-07-10/'
files = sorted(glob.glob(folder + '*.fits.gz'))

In [4]:
for file in files:
    process(file, dark_file=dark_file,
            _calc_wavelengths=True, _find_center=True, _mask=True, _rebin=8, to_file=True)

KeyboardInterrupt: 

In [5]:
files = sorted(glob.glob('*.fits'))

In [6]:
k_T = 0.03

wvlns = []
temperatures = []
datas = []
mus = []

for file in files:
    with fits.open(file) as hdul:
        data = hdul[0].data
        header = hdul[0].header

    nx, ny = header['NAXIS2'], header['NAXIS1']
    xc, yc = header['CRPIX2'] - 1, header['CRPIX1'] - 1
    rsun = header['RSUN_ARC'] / header['CDELT1']

    xi, yi = np.mgrid[:nx,:ny]
    mu = np.sqrt((rsun ** 2 - (xi - xc) ** 2 - (yi - yc) ** 2).clip(0)) / rsun

    datas += [data]
    wvlns += [read_wavelengths(header)]
    temperatures += [header['FGOV1PT1']]
    mus += [mu]

datas = np.swapaxes(np.concat(datas, axis=1), 0, 1)
wvlns = np.array(wvlns)
temperatures = np.array(temperatures)
mus = np.array(mus)

wvlns -= k_T * (np.expand_dims(temperatures, -1) - 61)

In [7]:
dwv = 0.01

a, b = np.max(np.min(wvlns, axis=-1)), np.min(np.max(wvlns, axis=-1))
a = np.round(a + dwv / 2, 2)
b = np.round(b - dwv / 2, 2)

wv = np.arange(a, b + dwv / 2, dwv)

In [8]:
from fitting import interpolate

Q = []
for i in range(3):
    Q += [interpolate(datas[i], wvlns[i], np.expand_dims(wv, (-1,-2)))]
Q = np.array(Q)

In [9]:
x, y = 50, 55

plt.figure(figsize=(10,8))
for i in range(3):
    plt.plot(wv, Q[i,:,x,y])
plt.tight_layout()

In [10]:
params, gamma = fit_prefilter(Q[0], wv, np.expand_dims(mus[0], 0),
                   axis=0, lam=0, niter=5)

In [11]:
gamma

array([ 0.06432204,  0.04555157,  0.05517767, -0.01003825])

In [15]:
plt.figure(figsize=(10,10))
plt.imshow(params[0], 'seismic')
plt.tight_layout()

In [16]:
plt.figure(figsize=(10,10))
plt.imshow(params[1], 'inferno')
plt.tight_layout()

In [138]:
plt.figure(figsize=(10,10))
plt.imshow(params[1,2], 'inferno', vmin=0.035, vmax=0.055)
plt.tight_layout()

In [ ]:
data.shape